## Cell 0: setup, data and label matrices

In [10]:
# 05_probe - Cell 0: setup, data and label matrices
from pathlib import Path
import json, time, warnings
import yaml
import numpy as np
import pandas as pd
from itertools import product

PROJECT_ROOT = Path.cwd().parent.parent
with open(PROJECT_ROOT / "configs" / "paths.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
with open(PROJECT_ROOT / "configs" / "probe.yaml", encoding="utf-8") as f:
    P = yaml.safe_load(f)

D = PROJECT_ROOT / cfg["outputs"]["processed_dir"] / "dataset_v1"
EMB = D / P["embeddings"]["dir"]

# RES = D / "probe_results"; RES.mkdir(exist_ok=True)

FLOOR = 30                                         # 50 = main probe (already done); 40 and 30 = sensitivity
RES = D / "probe_results" / ("" if FLOOR == 50 else f"floor{FLOOR}"); RES.mkdir(parents=True, exist_ok=True)

K, SEEDS, N_BOOT = P["evaluation"]["k"], P["evaluation"]["seeds"], P["evaluation"]["bootstrap"]
C_GRID = P["model"]["C_grid"]
POP_BASE = P["baselines"]["disease_popularity"]["degree_matched"]
N_JOBS, W = -1, 1022

B = lambda cols: {c: bool for c in cols}
circ = pd.read_csv(D / "circrnas.csv", keep_default_na=False, dtype=B(["mirna_labelled", "rbp_labelled", "is_hub"]))
folds = pd.read_csv(D / "folds.csv")
emb_index = pd.read_csv(EMB / "rnafm_embeddings_index.csv")
for t, name in [(folds, "folds.csv"), (emb_index, "embedding index")]:
    assert t["circbase_id"].tolist() == circ["circbase_id"].tolist(), f"{name}: row order differs from circrnas.csv"
ids, groups = circ["circbase_id"].to_numpy(), circ["host_group"].to_numpy()
seq_len = circ["seq_len"].to_numpy()
LONG, HUB = seq_len > W, circ["is_hub"].to_numpy()

X_ov = np.load(EMB / P["embeddings"]["main"]).astype(np.float64)
X_no = np.load(EMB / P["embeddings"]["sensitivity"]).astype(np.float64)
loglen = np.log10(seq_len)[:, None].astype(np.float64)
assert X_ov.shape == X_no.shape and X_ov.shape[0] == len(circ)
INPUTS = {"length": loglen, "embedding": X_ov, "embedding+length": np.hstack([X_ov, loglen]),
          "embedding_nonoverlap": X_no}

# RBP / miRNA label matrices: rows = all circRNAs (unlabelled rows masked), columns = head targets
mt = pd.read_csv(D / "mirna_targets.csv", dtype=B(["in_head"]))
rt = pd.read_csv(D / "rbp_targets.csv", keep_default_na=False, dtype=B(["in_head", "hub"]))
mir = pd.read_csv(D / "mirna_edges.csv", dtype=B(["pos_working"]))
rbp = pd.read_csv(D / "rbp_edges.csv", keep_default_na=False, dtype=B(["pos_working"]))

def label_matrix(edges, key, targets):
    pos = edges[edges["pos_working"] & edges[key].isin(targets)].drop_duplicates(["circbase_id", key])
    Y = pd.crosstab(pos["circbase_id"], pos[key]).reindex(index=ids, columns=targets, fill_value=0) > 0
    return Y.to_numpy().astype(np.int8)

AUX = {}
for head, edges, key, tt, lab in [("miRNA", mir, "mirna_id", mt, "mirna_labelled"),
                                  ("RBP", rbp, "rbp", rt, "rbp_labelled")]:
    targets = sorted(tt.loc[tt["n_pos"] >= FLOOR, key])
    Y, mask = label_matrix(edges, key, targets), circ[lab].to_numpy()
    assert Y[~mask].sum() == 0, f"{head}: positives on unlabelled circRNAs"
    assert (Y.sum(0) == tt.set_index(key).loc[targets, "n_pos"].to_numpy()).all(), f"{head}: matrix != target table"
    hub_cols = np.isin(targets, rt.loc[rt["hub"], "rbp"]) if head == "RBP" else np.zeros(len(targets), bool)
    AUX[head] = {"Y": Y, "mask": mask, "targets": targets, "hub_cols": hub_cols}
    print(f"{head}: {len(targets)} targets | labelled circRNAs {mask.sum()} | positives {int(Y.sum())} | hub targets {hub_cols.sum()}")


seqs = pd.read_csv(D / "sequences.csv", keep_default_na=False)
assert seqs["circbase_id"].tolist() == circ["circbase_id"].tolist(), "sequences.csv row order differs"

def kmer_freq(seqs_list, k):
    vocab = {"".join(p): i for i, p in enumerate(product("ACGU", repeat=k))}
    M = np.zeros((len(seqs_list), len(vocab)))
    for n, s in enumerate(seqs_list):
        s2 = s + s[:k - 1]                                  # wrap: k-mers across the back-splice junction
        for i in range(len(s)):
            j = vocab.get(s2[i:i + k])                      # k-mers with N or other letters are skipped
            if j is not None:
                M[n, j] += 1
    return M / np.maximum(M.sum(1, keepdims=True), 1)       # frequencies: independent of length

X_kmer = np.hstack([kmer_freq(seqs["sequence"].tolist(), 3), kmer_freq(seqs["sequence"].tolist(), 4)])
assert X_kmer.shape == (len(circ), 320) and np.allclose(X_kmer[:, :64].sum(1), 1)
INPUTS["kmer"] = X_kmer
print("k-mer features:", X_kmer.shape)

INPUTS["embedding_L6"] = np.load(EMB / P["embeddings"]["layer6"]).astype(np.float64)
assert INPUTS["embedding_L6"].shape == X_ov.shape

if FLOOR != 50:                                    # sensitivity runs: only the inputs the floor question needs
    INPUTS = {k: INPUTS[k] for k in ["length", "embedding", "kmer"]}

# Disease pairs and frozen negatives, as integer indices
cda = pd.read_csv(D / "cda_pairs.csv")
dis = pd.read_csv(D / "diseases.csv")
negs = pd.read_csv(D / "negatives_cda.csv")
D_ID, C_ID = dict(zip(dis["disease"], dis["disease_id"])), {c: i for i, c in enumerate(ids)}
for t in (cda, negs):
    t["c"], t["d"] = t["circbase_id"].map(C_ID), t["disease"].map(D_ID)
    assert t[["c", "d"]].notna().all().all(), "unknown circRNA or disease"
    t[["c", "d"]] = t[["c", "d"]].astype(int)
N_DIS = len(dis)
print(f"disease: {len(cda)} pairs | {N_DIS} diseases | frozen negatives {len(negs)}")
print(f"inputs: { {k: v.shape[1] for k, v in INPUTS.items()} } | results -> {RES}")

miRNA: 130 targets | labelled circRNAs 1433 | positives 5655 | hub targets 0
RBP: 103 targets | labelled circRNAs 1405 | positives 21518 | hub targets 5
k-mer features: (1582, 320)
disease: 2137 pairs | 207 diseases | frozen negatives 12822
inputs: {'length': 1, 'embedding': 640, 'kmer': 320} | results -> d:\Queens_university\Project\Working\Notebook\circmtl\data\processed\dataset_v1\probe_results\floor30


## Cell 1: RBP and miRNA probes (the long run, cached)

In [11]:
# Cell 1 - RBP and miRNA probes: one L2 logistic regression per target, C chosen by inner grouped CV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from joblib import Parallel, delayed
from scipy.stats import rankdata

def _fit_predict(Xtr, ytr, Xte, inner):
    if ytr.sum() < 3:                                    # too few positives in this training fold
        return np.full(len(Xte), np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            m = LogisticRegressionCV(Cs=C_GRID, cv=inner, scoring="roc_auc", max_iter=5000).fit(Xtr, ytr)
            if not np.isfinite(m.C_).all():
                raise ValueError("no valid C")
        except ValueError:                               # an inner fold without both classes
            m = LogisticRegression(C=1.0, max_iter=5000).fit(Xtr, ytr)
    return m.decision_function(Xte)

def probe_matrix(X, Y, mask, seed):
    """Out-of-fold scores for every column of Y, on rows in mask; scaler and C fitted on training folds only."""
    rows = np.flatnonzero(mask)
    oof = np.full(Y.shape, np.nan, dtype=np.float32)
    fcol = folds[f"fold_seed{seed}"].to_numpy()
    for f in range(K):
        tr, te = rows[fcol[rows] != f], rows[fcol[rows] == f]
        sc = StandardScaler().fit(X[tr])
        Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        inner = list(GroupKFold(n_splits=3).split(Xtr, groups=groups[tr]))
        cols = Parallel(n_jobs=N_JOBS)(delayed(_fit_predict)(Xtr, Y[tr, j], Xte, inner) for j in range(Y.shape[1]))
        # Rank-normalise within the test fold: removes fold-specific intercepts/scales that bias pooled AUROC
        oof[te] = np.column_stack([rankdata(c) / len(c) if np.isfinite(c).all() else c for c in cols])
    return oof

t0 = time.time()
for head in ("miRNA", "RBP"):
    A = AUX[head]
    for name, X in INPUTS.items():
        for s in SEEDS:
            fn = RES / f"oof_{head}_{name}_seed{s}.npy"
            if fn.exists():
                continue                                 # cached: delete the file to recompute
            np.save(fn, probe_matrix(X, A["Y"], A["mask"], s))
            print(f"  {head:5s} {name:22s} seed {s}  {time.time() - t0:6.0f}s")
print(f"RBP/miRNA probes done in {time.time() - t0:.0f}s")

  miRNA length                 seed 0       6s
  miRNA length                 seed 1      11s
  miRNA length                 seed 2      17s
  miRNA embedding              seed 0     148s
  miRNA embedding              seed 1     277s
  miRNA embedding              seed 2     404s
  miRNA kmer                   seed 0     428s
  miRNA kmer                   seed 1     455s
  miRNA kmer                   seed 2     480s
  RBP   length                 seed 0     484s
  RBP   length                 seed 1     488s
  RBP   length                 seed 2     492s
  RBP   embedding              seed 0     668s
  RBP   embedding              seed 1     848s
  RBP   embedding              seed 2    1037s
  RBP   kmer                   seed 0    1058s
  RBP   kmer                   seed 1    1078s
  RBP   kmer                   seed 2    1099s
RBP/miRNA probes done in 1099s


## Cell 2: RBP and miRNA results, bootstrap and pass rule

In [12]:
# Cell 2 - RBP/miRNA metrics, bootstrap over host groups, pass rule
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score

def load_oof(head, name, seeds=SEEDS):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                  # all-NaN rows are the masked circRNAs
        return np.nanmean(np.stack([np.load(RES / f"oof_{head}_{name}_seed{s}.npy") for s in seeds]), axis=0)

def fast_auroc(Y, S):
    """Column-wise AUROC (ties averaged). Y, S: (rows, targets) without NaN."""
    R = rankdata(S, axis=0)
    n1 = Y.sum(0); n0 = len(Y) - n1
    with np.errstate(invalid="ignore", divide="ignore"):
        return ((R * Y).sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)

def target_metrics(head, name, rows, seeds=SEEDS):
    Y, O = AUX[head]["Y"][rows], load_oof(head, name, seeds)[rows]
    ok = ~np.isnan(O).any(0)
    au = np.full(Y.shape[1], np.nan); ap = np.full(Y.shape[1], np.nan)
    au[ok] = fast_auroc(Y[:, ok], O[:, ok])
    for j in np.flatnonzero(ok):
        if 0 < Y[:, j].sum() < len(Y): ap[j] = average_precision_score(Y[:, j], O[:, j])
    return au, ap

def boot_aux(head, a, b, rows, cols=None, n=N_BOOT, seed=0):
    """Median-over-targets AUROC of inputs a and b, and their difference, resampling host groups."""
    Y, Oa, Ob = AUX[head]["Y"], load_oof(head, a), load_oof(head, b)
    cols = np.ones(Y.shape[1], bool) if cols is None else cols
    ug, inv = np.unique(groups[rows], return_inverse=True)
    members = [rows[inv == k] for k in range(len(ug))]
    rng, S = np.random.default_rng(seed), []
    for _ in range(n):
        rr = np.concatenate([members[k] for k in rng.integers(0, len(ug), len(ug))])
        y = Y[rr][:, cols]
        keep = (y.sum(0) > 0) & (y.sum(0) < len(rr))
        ma = np.nanmedian(fast_auroc(y[:, keep], Oa[rr][:, cols][:, keep]))
        mb = np.nanmedian(fast_auroc(y[:, keep], Ob[rr][:, cols][:, keep]))
        S.append((ma, mb, ma - mb))
    S = np.array(S)
    point_a = np.nanmedian(target_metrics(head, a, rows)[0][cols]); point_b = np.nanmedian(target_metrics(head, b, rows)[0][cols])
    lo, hi = np.percentile(S[:, 2], [2.5, 97.5])
    return {"head": head, "a": a, "b": b, "median_auroc_a": round(point_a, 3), "median_auroc_b": round(point_b, 3),
            "diff": round(point_a - point_b, 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3), "n_rows": len(rows)}

# 1. Summary table: median over targets, for all rows and by length
rows_aux = []
for head in ("miRNA", "RBP"):
    A = AUX[head]    
    for name in INPUTS:
        for subset, m in [("all", A["mask"]), ("<=1022 nt", A["mask"] & ~LONG), (">1022 nt", A["mask"] & LONG)]:
            au, ap = target_metrics(head, name, np.flatnonzero(m))
            per_seed = [np.nanmedian(target_metrics(head, name, np.flatnonzero(m), [s])[0]) for s in SEEDS]
            rows_aux.append({"head": head, "input": name, "subset": subset, "median_AUROC": round(np.nanmedian(au), 3),
                             "median_AUPR": round(np.nanmedian(ap), 3), "seed_range": f"{min(per_seed):.3f}-{max(per_seed):.3f}",
                             "targets_scored": int(np.isfinite(au).sum())})
T_aux = pd.DataFrame(rows_aux)
print(T_aux.to_string(index=False))

# 2. Bootstrap comparisons (CI of the difference in median AUROC)

# 2. Bootstrap comparisons (CI of the difference in median AUROC)
comp = []
for head in ("miRNA", "RBP"):
    A = AUX[head]
    allr, longr = np.flatnonzero(A["mask"]), np.flatnonzero(A["mask"] & LONG)
    pairs_to_test = [("embedding", "length", allr), ("embedding+length", "length", allr),
                     ("embedding", "embedding_nonoverlap", longr), ("embedding", "kmer", allr),
                     ("embedding_L6", "length", allr), ("embedding_L6", "embedding", allr)]
    for a, b, rows in pairs_to_test:
        if a in INPUTS and b in INPUTS:
            comp.append(boot_aux(head, a, b, rows))
    if A["hub_cols"].any():
        r = boot_aux(head, "embedding", "length", allr, cols=~A["hub_cols"]); r["a"] += " (no hub RBPs)"; comp.append(r)
C_aux = pd.DataFrame(comp)
print("\n" + C_aux.to_string(index=False))

# comp = []
# for head in ("miRNA", "RBP"):
#     A = AUX[head]
#     allr, longr = np.flatnonzero(A["mask"]), np.flatnonzero(A["mask"] & LONG)
#     comp.append(boot_aux(head, "embedding", "length", allr))
#     comp.append(boot_aux(head, "embedding+length", "length", allr))
#     comp.append(boot_aux(head, "embedding", "embedding_nonoverlap", longr))
#     comp.append(boot_aux(head, "embedding", "kmer", allr))
#     comp.append(boot_aux(head, "embedding_L6", "length", allr))
#     comp.append(boot_aux(head, "embedding_L6", "embedding", allr))       # L6 - L12
#     if A["hub_cols"].any():
#         r = boot_aux(head, "embedding", "length", allr, cols=~A["hub_cols"]); r["a"] += " (no hub RBPs)"; comp.append(r)
# C_aux = pd.DataFrame(comp)
# print("\n" + C_aux.to_string(index=False))

# 3. Pass rule (fixed in probe.yaml): embedding beats length, CI of the difference excludes 0
for head in ("miRNA", "RBP"):
    r = C_aux[(C_aux["head"] == head) & (C_aux["a"] == "embedding") & (C_aux["b"] == "length")].iloc[0]
    print(f"{head}: embedding {r.median_auroc_a} vs length {r.median_auroc_b} | diff {r['diff']} "
          f"[{r.ci_low}, {r.ci_high}] -> {'PASS' if r.ci_low > 0 else 'NOT PASSED'}")
T_aux.to_csv(RES / "aux_summary.csv", index=False); C_aux.to_csv(RES / "aux_bootstrap.csv", index=False)

 head     input    subset  median_AUROC  median_AUPR  seed_range  targets_scored
miRNA    length       all         0.745        0.095 0.745-0.745             130
miRNA    length <=1022 nt         0.637        0.033 0.633-0.644             130
miRNA    length  >1022 nt         0.633        0.128 0.629-0.638             130
miRNA embedding       all         0.735        0.085 0.718-0.734             130
miRNA embedding <=1022 nt         0.681        0.042 0.657-0.669             130
miRNA embedding  >1022 nt         0.639        0.131 0.624-0.650             130
miRNA      kmer       all         0.675        0.057 0.662-0.679             130
miRNA      kmer <=1022 nt         0.686        0.033 0.666-0.683             130
miRNA      kmer  >1022 nt         0.665        0.128 0.641-0.665             130
  RBP    length       all         0.663        0.175 0.661-0.663             103
  RBP    length <=1022 nt         0.617        0.109 0.614-0.619             103
  RBP    length  >1022 nt   

In [ ]:
# Cell 2 - RBP/miRNA metrics, bootstrap over host groups, pass rule
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score

def load_oof(head, name, seeds=SEEDS):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                  # all-NaN rows are the masked circRNAs
        return np.nanmean(np.stack([np.load(RES / f"oof_{head}_{name}_seed{s}.npy") for s in seeds]), axis=0)

def fast_auroc(Y, S):
    """Column-wise AUROC (ties averaged). Y, S: (rows, targets) without NaN."""
    R = rankdata(S, axis=0)
    n1 = Y.sum(0); n0 = len(Y) - n1
    with np.errstate(invalid="ignore", divide="ignore"):
        return ((R * Y).sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)

def target_metrics(head, name, rows, seeds=SEEDS):
    Y, O = AUX[head]["Y"][rows], load_oof(head, name, seeds)[rows]
    ok = ~np.isnan(O).any(0)
    au = np.full(Y.shape[1], np.nan); ap = np.full(Y.shape[1], np.nan)
    au[ok] = fast_auroc(Y[:, ok], O[:, ok])
    for j in np.flatnonzero(ok):
        if 0 < Y[:, j].sum() < len(Y): ap[j] = average_precision_score(Y[:, j], O[:, j])
    return au, ap

def boot_aux(head, a, b, rows, cols=None, n=N_BOOT, seed=0):
    """Median-over-targets AUROC of inputs a and b, and their difference, resampling host groups."""
    Y, Oa, Ob = AUX[head]["Y"], load_oof(head, a), load_oof(head, b)
    cols = np.ones(Y.shape[1], bool) if cols is None else cols
    ug, inv = np.unique(groups[rows], return_inverse=True)
    members = [rows[inv == k] for k in range(len(ug))]
    rng, S = np.random.default_rng(seed), []
    for _ in range(n):
        rr = np.concatenate([members[k] for k in rng.integers(0, len(ug), len(ug))])
        y = Y[rr][:, cols]
        keep = (y.sum(0) > 0) & (y.sum(0) < len(rr))
        ma = np.nanmedian(fast_auroc(y[:, keep], Oa[rr][:, cols][:, keep]))
        mb = np.nanmedian(fast_auroc(y[:, keep], Ob[rr][:, cols][:, keep]))
        S.append((ma, mb, ma - mb))
    S = np.array(S)
    point_a = np.nanmedian(target_metrics(head, a, rows)[0][cols]); point_b = np.nanmedian(target_metrics(head, b, rows)[0][cols])
    lo, hi = np.percentile(S[:, 2], [2.5, 97.5])
    return {"head": head, "a": a, "b": b, "median_auroc_a": round(point_a, 3), "median_auroc_b": round(point_b, 3),
            "diff": round(point_a - point_b, 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3), "n_rows": len(rows)}

# 1. Summary table: median over targets, for all rows and by length
rows_aux = []
for head in ("miRNA", "RBP"):
    A = AUX[head]    
    for name in INPUTS:
        for subset, m in [("all", A["mask"]), ("<=1022 nt", A["mask"] & ~LONG), (">1022 nt", A["mask"] & LONG)]:
            au, ap = target_metrics(head, name, np.flatnonzero(m))
            per_seed = [np.nanmedian(target_metrics(head, name, np.flatnonzero(m), [s])[0]) for s in SEEDS]
            rows_aux.append({"head": head, "input": name, "subset": subset, "median_AUROC": round(np.nanmedian(au), 3),
                             "median_AUPR": round(np.nanmedian(ap), 3), "seed_range": f"{min(per_seed):.3f}-{max(per_seed):.3f}",
                             "targets_scored": int(np.isfinite(au).sum())})
T_aux = pd.DataFrame(rows_aux)
print(T_aux.to_string(index=False))

# 2. Bootstrap comparisons (CI of the difference in median AUROC)
comp = []
for head in ("miRNA", "RBP"):
    A = AUX[head]
    allr, longr = np.flatnonzero(A["mask"]), np.flatnonzero(A["mask"] & LONG)
    pairs_to_test = [("embedding", "length", allr), ("embedding+length", "length", allr),
                     ("embedding", "embedding_nonoverlap", longr), ("embedding", "kmer", allr),
                     ("embedding_L6", "length", allr), ("embedding_L6", "embedding", allr)]
    for a, b, rows in pairs_to_test:
        if a in INPUTS and b in INPUTS:
            comp.append(boot_aux(head, a, b, rows))
    if A["hub_cols"].any():
        r = boot_aux(head, "embedding", "length", allr, cols=~A["hub_cols"]); r["a"] += " (no hub RBPs)"; comp.append(r)
C_aux = pd.DataFrame(comp)
print("\n" + C_aux.to_string(index=False))

# 2. Bootstrap comparisons (CI of the difference in median AUROC)
# comp = []
# for head in ("miRNA", "RBP"):
#     A = AUX[head]
#     allr, longr = np.flatnonzero(A["mask"]), np.flatnonzero(A["mask"] & LONG)
#     comp.append(boot_aux(head, "embedding", "length", allr))
#     comp.append(boot_aux(head, "embedding+length", "length", allr))
#     comp.append(boot_aux(head, "embedding", "embedding_nonoverlap", longr))
#     comp.append(boot_aux(head, "embedding", "kmer", allr))
#     comp.append(boot_aux(head, "embedding_L6", "length", allr))
#     comp.append(boot_aux(head, "embedding_L6", "embedding", allr))       # L6 - L12
#     if A["hub_cols"].any():
#         r = boot_aux(head, "embedding", "length", allr, cols=~A["hub_cols"]); r["a"] += " (no hub RBPs)"; comp.append(r)
# C_aux = pd.DataFrame(comp)
# print("\n" + C_aux.to_string(index=False))

# 3. Pass rule (fixed in probe.yaml): embedding beats length, CI of the difference excludes 0
for head in ("miRNA", "RBP"):
    r = C_aux[(C_aux["head"] == head) & (C_aux["a"] == "embedding") & (C_aux["b"] == "length")].iloc[0]
    print(f"{head}: embedding {r.median_auroc_a} vs length {r.median_auroc_b} | diff {r['diff']} "
          f"[{r.ci_low}, {r.ci_high}] -> {'PASS' if r.ci_low > 0 else 'NOT PASSED'}")
T_aux.to_csv(RES / "aux_summary.csv", index=False); C_aux.to_csv(RES / "aux_bootstrap.csv", index=False)

 head     input    subset  median_AUROC  median_AUPR  seed_range  targets_scored
miRNA    length       all         0.745        0.095 0.745-0.745             130
miRNA    length <=1022 nt         0.637        0.033 0.633-0.644             130
miRNA    length  >1022 nt         0.633        0.128 0.629-0.638             130
miRNA embedding       all         0.735        0.085 0.718-0.734             130
miRNA embedding <=1022 nt         0.681        0.042 0.657-0.669             130
miRNA embedding  >1022 nt         0.639        0.131 0.624-0.650             130
miRNA      kmer       all         0.675        0.057 0.662-0.679             130
miRNA      kmer <=1022 nt         0.686        0.033 0.666-0.683             130
miRNA      kmer  >1022 nt         0.665        0.128 0.641-0.665             130
  RBP    length       all         0.663        0.175 0.661-0.663             103
  RBP    length <=1022 nt         0.617        0.109 0.614-0.619             103
  RBP    length  >1022 nt   

## Cell 3: disease bilinear probe

In [4]:
# Cell 3 - disease bilinear probe: score(c, d) = (A x_c) . v_d + b_d, L2, fitted with L-BFGS
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANK = P["disease_probe"]["rank"]; NEG_PER_POS = P["disease_probe"]["negatives_per_positive_train"]
LAMBDAS = P["disease_probe"]["lambdas"]
pos_by_c = cda.groupby("c")["d"].apply(set).to_dict()

def sample_negs(c_pos, pop, rng, k):
    """k degree-matched negatives per positive (same circRNA, disease ~ training popularity, never a positive)."""
    c = np.repeat(c_pos, k); d = rng.choice(N_DIS, size=len(c), p=pop)
    for _ in range(100):
        bad = np.array([dd in pos_by_c[cc] for cc, dd in zip(c, d)])
        if not bad.any(): break
        d[bad] = rng.choice(N_DIS, size=int(bad.sum()), p=pop)
    keep = ~np.array([dd in pos_by_c[cc] for cc, dd in zip(c, d)])
    return c[keep], d[keep]

def training_pairs(pp, k, rng):
    cnt = np.bincount(pp["d"], minlength=N_DIS).astype(float)
    cn, dn = sample_negs(pp["c"].to_numpy(), cnt / cnt.sum(), rng, k)
    return (np.r_[pp["c"].to_numpy(), cn], np.r_[pp["d"].to_numpy(), dn],
            np.r_[np.ones(len(pp)), np.zeros(len(cn))])

def fit_bilinear(Xs, c, d, y, lam, seed):
    n_in = Xs.shape[1]; r = min(RANK, n_in)
    rng = np.random.default_rng(seed)
    shapes = [(r, n_in), (N_DIS, r), (N_DIS,)]; sizes = [int(np.prod(s)) for s in shapes]
    def unpack(th):
        A = th[:sizes[0]].reshape(shapes[0]); V = th[sizes[0]:sizes[0] + sizes[1]].reshape(shapes[1])
        return A, V, th[sizes[0] + sizes[1]:]
    uc, ci = np.unique(c, return_inverse=True); Xu = Xs[uc]                    # project each circRNA once
    wt = np.where(y == 1, 0.5 / y.sum(), 0.5 / (len(y) - y.sum()))            # balanced classes
    def f(th):
        A, V, b = unpack(th)
        Hu = Xu @ A.T; H, Vd = Hu[ci], V[d]
        z = (H * Vd).sum(1) + b[d]
        loss = (wt * (np.logaddexp(0, z) - y * z)).sum() + lam * ((A ** 2).sum() + (V ** 2).sum())
        g = wt * (expit(z) - y)
        G = np.zeros_like(Hu); np.add.at(G, ci, g[:, None] * Vd)
        gA = G.T @ Xu + 2 * lam * A
        gV = np.zeros_like(V); np.add.at(gV, d, g[:, None] * H); gV += 2 * lam * V
        gb = np.bincount(d, weights=g, minlength=N_DIS)
        return loss, np.concatenate([gA.ravel(), gV.ravel(), gb])
    th0 = np.concatenate([rng.normal(0, 1 / np.sqrt(n_in), sizes[0]), rng.normal(0, 1 / np.sqrt(r), sizes[1]), np.zeros(N_DIS)])
    A, V, b = unpack(minimize(f, th0, jac=True, method="L-BFGS-B", options={"maxiter": 500}).x)
    return lambda cq, dq: ((Xs[cq] @ A.T) * V[dq]).sum(1) + b[dq]

def probe_disease(X, seed):
    fcol = folds[f"fold_seed{seed}"].to_numpy(); pair_fold = fcol[cda["c"].to_numpy()]
    out = []
    for f in range(K):
        Xs = StandardScaler().fit(X[fcol != f]).transform(X)                   # scaler from training circRNAs only
        trp, tep = cda[pair_fold != f], cda[pair_fold == f]
        rng = np.random.default_rng(1000 * seed + f)
        it, iv = next(GroupShuffleSplit(1, test_size=0.2, random_state=1000 * seed + f)
                      .split(trp, groups=groups[trp["c"].to_numpy()]))
        ctr, dtr, ytr = training_pairs(trp.iloc[it], NEG_PER_POS, rng)
        cva, dva, yva = training_pairs(trp.iloc[iv], 1, rng)
        val = {lam: roc_auc_score(yva, fit_bilinear(Xs, ctr, dtr, ytr, lam, seed)(cva, dva)) for lam in LAMBDAS}
        lam = max(val, key=val.get)
        c_all, d_all, y_all = training_pairs(trp, NEG_PER_POS, rng)              # refit on the whole training fold
        model = fit_bilinear(Xs, c_all, d_all, y_all, lam, seed)
        for scheme in ("degree_matched", "random"):
            ng = negs[(negs["seed"] == seed) & (negs["fold"] == f) & (negs["scheme"] == scheme)]
            for part, label in [(tep, 1), (ng, 0)]:
                out.append(pd.DataFrame({"seed": seed, "fold": f, "scheme": scheme, "c": part["c"].to_numpy(),
                                         "d": part["d"].to_numpy(), "y": label, "lambda": lam,
                                         "score": model(part["c"].to_numpy(), part["d"].to_numpy())}))
    out = pd.concat(out, ignore_index=True)
    # Rank-normalise within each test set (seed, fold, scheme): removes fold-specific shifts before pooling
    out["score_rank"] = out.groupby(["seed", "fold", "scheme"])["score"].rank(pct=True)
    return out

t0 = time.time()
for name, X in INPUTS.items():
    fn = RES / f"disease_{name}.csv"
    if fn.exists(): continue
    pd.concat([probe_disease(X, s) for s in SEEDS], ignore_index=True).to_csv(fn, index=False)
    print(f"  disease {name:22s} {time.time() - t0:6.0f}s")
print(f"disease probes done in {time.time() - t0:.0f}s")

  disease embedding_L6              699s
disease probes done in 699s


## Cell 4: disease results, bootstrap, pass rule and splits

In [5]:
# Cell 4 - disease metrics, bootstrap over host groups, pass rule, reported splits
DIS = {name: pd.read_csv(RES / f"disease_{name}.csv") for name in INPUTS}
keys = ["seed", "fold", "scheme", "c", "d", "y"]
for name in INPUTS:
    assert DIS[name][keys].equals(DIS["length"][keys]), f"{name}: rows differ from the length run"

def mean_auc(df, w=None):
    """AUROC per seed (folds pooled on within-fold ranks), averaged over seeds."""
    vals = []
    for s, g in df.groupby("seed"):
        ww = None if w is None else w[g.index]
        if ww is not None and (ww > 0).sum() == 0: continue
        vals.append(roc_auc_score(g["y"], g["score_rank"], sample_weight=ww))
    return float(np.mean(vals))

def boot_disease(a, b, scheme="degree_matched", subset=None, n=N_BOOT, seed=0):
    da = DIS[a][DIS[a]["scheme"] == scheme].reset_index(drop=True)
    db = DIS[b][DIS[b]["scheme"] == scheme].reset_index(drop=True)
    if subset is not None:
        keep = subset(da).to_numpy(); da, db = da[keep].reset_index(drop=True), db[keep].reset_index(drop=True)
    ug, gi = np.unique(groups[da["c"].to_numpy()], return_inverse=True)
    rng, S = np.random.default_rng(seed), []
    for _ in range(n):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[gi].astype(float)
        S.append((mean_auc(da, w), mean_auc(db, w)))
    S = np.array(S); d_ = S[:, 0] - S[:, 1]
    pa, pb = mean_auc(da), mean_auc(db)
    return {"a": a, "b": b, "scheme": scheme, "auroc_a": round(pa, 3), "ci_a": f"[{np.percentile(S[:,0],2.5):.3f}, {np.percentile(S[:,0],97.5):.3f}]",
            "auroc_b": round(pb, 3), "diff": round(pa - pb, 3),
            "ci_diff": f"[{np.percentile(d_,2.5):.3f}, {np.percentile(d_,97.5):.3f}]", "ci_a_low": np.percentile(S[:, 0], 2.5),
            "ci_diff_low": np.percentile(d_, 2.5), "n_pairs": len(da)}

# 1. Summary: AUROC and AUPR per input and scheme
rows_d = []
for name, df in DIS.items():
    for scheme in ("degree_matched", "random"):
        g = df[df["scheme"] == scheme]
        per = [roc_auc_score(x["y"], x["score_rank"]) for _, x in g.groupby("seed")]
        ap = np.mean([average_precision_score(x["y"], x["score_rank"]) for _, x in g.groupby("seed")])
        rows_d.append({"input": name, "scheme": scheme, "AUROC": round(np.mean(per), 3), "AUPR": round(ap, 3),
                       "seed_range": f"{min(per):.3f}-{max(per):.3f}", "lambda_mode": g["lambda"].mode().iat[0]})
T_dis = pd.DataFrame(rows_d)
print(T_dis.to_string(index=False))

# 2. Bootstrap comparisons and reported splits
diff31 = pd.read_csv(EMB / "chunking_comparison.csv")
most_diff = set(diff31.loc[diff31["cos_centred"] < 0.95, "circbase_id"].map(C_ID))
in_long = lambda d: pd.Series(LONG[d["c"].to_numpy()])
comps = [boot_disease("embedding", "length"),
         boot_disease("embedding+length", "length"),
         boot_disease("embedding", "length", subset=lambda d: pd.Series(~HUB[d["c"].to_numpy()])) | {"a": "embedding (no hubs)"},
         boot_disease("embedding", "length", subset=lambda d: pd.Series(~LONG[d["c"].to_numpy()])) | {"a": "embedding (<=1022 nt)"},
         boot_disease("embedding", "length", subset=in_long) | {"a": "embedding (>1022 nt)"},
         boot_disease("embedding", "embedding_nonoverlap", subset=in_long) | {"a": "overlap vs non-overlap (>1022 nt)"},
         boot_disease("embedding", "length", scheme="random"),
         boot_disease("embedding", "kmer"),
         boot_disease("embedding_L6", "length"),
         boot_disease("embedding_L6", "embedding")]                      # L6 - L12
C_dis = pd.DataFrame(comps)
print("\n" + C_dis.drop(columns=["ci_a_low", "ci_diff_low"]).to_string(index=False))

# 3. Pass rule: embedding > popularity baseline on degree-matched negatives (CI excludes it), and > length
r = C_dis.iloc[0]
print(f"\ndisease: embedding AUROC {r.auroc_a} {r.ci_a} vs popularity {POP_BASE} -> "
      f"{'PASS' if r.ci_a_low > POP_BASE else 'NOT PASSED'} | vs length: diff {r['diff']} {r.ci_diff} -> "
      f"{'PASS' if r.ci_diff_low > 0 else 'NOT PASSED'}")

# 4. Descriptive only: circRNAs whose two embeddings differ most (centred cosine < 0.95)
dm = DIS["embedding"]["scheme"] == "degree_matched"
sub = DIS["embedding"][dm]["c"].isin(most_diff)
if sub.sum() and DIS["embedding"][dm][sub]["y"].nunique() == 2:
    print(f"\n{len(most_diff)} most-different circRNAs ({int(sub.sum())} pairs, descriptive): "
          f"overlap AUROC {mean_auc(DIS['embedding'][dm][sub].reset_index(drop=True)):.3f} | "
          f"non-overlap {mean_auc(DIS['embedding_nonoverlap'][dm][sub].reset_index(drop=True)):.3f}")
T_dis.to_csv(RES / "disease_summary.csv", index=False); C_dis.to_csv(RES / "disease_bootstrap.csv", index=False)

# Layer decision (probe.yaml: layer_decision)
g = lambda T, h, a, b: T[(T["head"] == h) & (T["a"] == a) & (T["b"] == b)].iloc[0]
m_pass = g(C_aux, "miRNA", "embedding_L6", "length")["ci_low"] > 0
r_ok = g(C_aux, "RBP", "embedding_L6", "embedding")["ci_high"] >= 0
d_row = C_dis[(C_dis["a"] == "embedding_L6") & (C_dis["b"] == "embedding")].iloc[0]
d_ok = float(d_row["ci_diff"].strip("[]").split(",")[1]) >= 0
print(f"miRNA passes with L6: {m_pass} | RBP not worse: {r_ok} | disease not worse: {d_ok}")
print("-> use LAYER 6 for all tasks" if (m_pass and r_ok and d_ok) else "-> keep LAYER 12; report miRNA as not passed")

               input         scheme  AUROC  AUPR  seed_range  lambda_mode
              length degree_matched  0.527 0.514 0.515-0.540       0.0001
              length         random  0.591 0.515 0.584-0.602       0.0001
           embedding degree_matched  0.529 0.520 0.526-0.532       0.0100
           embedding         random  0.536 0.522 0.526-0.553       0.0100
    embedding+length degree_matched  0.530 0.520 0.526-0.532       0.0100
    embedding+length         random  0.531 0.519 0.526-0.535       0.0100
embedding_nonoverlap degree_matched  0.531 0.521 0.525-0.541       0.0010
embedding_nonoverlap         random  0.521 0.516 0.503-0.536       0.0010
                kmer degree_matched  0.519 0.513 0.509-0.529       0.0100
                kmer         random  0.519 0.506 0.499-0.536       0.0100
        embedding_L6 degree_matched  0.530 0.521 0.529-0.531       0.0100
        embedding_L6         random  0.534 0.528 0.477-0.566       0.0100

                                a    

## Cell 5: check with per-disease classifiers (diseases with ≥10 circRNAs)

In [6]:
# Cell 5 - check: per-disease logistic regression for diseases with >= 10 circRNAs (all circRNAs as rows)
n_c = cda.groupby("d")["c"].nunique()
big = n_c[n_c >= 10].index.to_numpy()
Yd = np.zeros((len(circ), len(big)), dtype=np.int8)
for j, dd in enumerate(big):
    Yd[cda.loc[cda["d"] == dd, "c"].to_numpy(), j] = 1
AUX["disease_LR"] = {"Y": Yd, "mask": np.ones(len(circ), bool), "targets": list(dis.set_index("disease_id").loc[big, "disease"]),
                     "hub_cols": np.zeros(len(big), bool)}
for name in ("length", "embedding"):
    for s in SEEDS:
        fn = RES / f"oof_disease_LR_{name}_seed{s}.npy"
        if not fn.exists():
            np.save(fn, probe_matrix(INPUTS[name], Yd, AUX["disease_LR"]["mask"], s))
allr = np.arange(len(circ))
r = boot_aux("disease_LR", "embedding", "length", allr)
print(f"per-disease LR on {len(big)} diseases with >= 10 circRNAs: median AUROC embedding {r['median_auroc_a']} "
      f"vs length {r['median_auroc_b']} | diff {r['diff']} [{r['ci_low']}, {r['ci_high']}]")
au_e = target_metrics("disease_LR", "embedding", allr)[0]
print(pd.DataFrame({"disease": AUX["disease_LR"]["targets"], "n_circRNAs": n_c[big].to_numpy(), "AUROC_embedding": au_e.round(3)})
      .sort_values("AUROC_embedding", ascending=False).to_string(index=False))

per-disease LR on 51 diseases with >= 10 circRNAs: median AUROC embedding 0.523 vs length 0.549 | diff -0.025 [-0.055, 0.015]
                           disease  n_circRNAs  AUROC_embedding
                 Uterine leiomyoma          21            0.754
                 Pancreatic cancer          10            0.733
                 Esophageal cancer          14            0.697
 Laryngeal squamous cell carcinoma          14            0.690
               Alzheimer's disease          12            0.688
                Multiple sclerosis          11            0.662
   Proliferative vitreoretinopathy          15            0.659
   Clear cell renal cell carcinoma          17            0.651
      Oral squamous cell carcinoma          27            0.643
            Pulmonary tuberculosis          13            0.626
           Coronary artery disease          14            0.625
     Triple negative breast cancer          19            0.624
  Intervertebral disc degeneration        